# Unit 2 · From Text to Tensors
**Learn with Adi · Build an LLM from Scratch**

Executable twin of [Unit 2](https://aditya-402.github.io/learn-with-adi/series/llm-from-scratch/unit2.html) —
the whole input pipeline: text → tokens → token IDs → embeddings. Run top to bottom.


In [ ]:
# one-time setup (Colab): the BPE tokenizer library
!pip install -q tiktoken

## The training text — a 20,479-character public-domain short story

In [ ]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/"
       "ch02/01_main-chapter-code/the-verdict.txt")
urllib.request.urlretrieve(url, "the-verdict.txt")
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("characters:", len(raw_text))
print(raw_text[:99])

## Splitting text into tokens — the regex we settle on

In [ ]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("tokens:", len(preprocessed))   # 4,690
print(preprocessed[:30])

## Vocabulary + a tokenizer class with encode/decode

In [ ]:
all_words = sorted(set(preprocessed))
vocab = {token: integer for integer, token in enumerate(all_words)}
print("vocabulary size:", len(vocab))   # 1,130

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        return [self.str_to_int[s] for s in preprocessed]
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

tok = SimpleTokenizerV1(vocab)
ids = tok.encode('"It\'s the last he painted, you know," Mrs. Gisburn said')
print(ids)
print(tok.decode(ids))

## Special tokens — surviving unknown words and document boundaries

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print("vocabulary size now:", len(vocab))   # 1,132

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

t1 = "Hello, do you like tea?"
t2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((t1, t2))
tok2 = SimpleTokenizerV2(vocab)
print(tok2.decode(tok2.encode(text)))   # Hello -> <|unk|>

## Byte pair encoding — the production tokenizer

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
ids = tokenizer.encode("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.",
                       allowed_special={"<|endoftext|>"})
print(ids)
print(tokenizer.decode(ids))   # nothing is unknown to BPE — it falls back to subwords

## The sliding window — a dataset that labels itself
Expected first batch inputs: `[[40, 367, 2885, 1464]]`, targets `[[367, 2885, 1464, 1807]]`

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))
    def __len__(self): return len(self.input_ids)
    def __getitem__(self, idx): return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=num_workers)

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
first = next(iter(dataloader))
print("inputs: ", first[0])
print("targets:", first[1])

## Token + positional embeddings — the tensor Unit 3 receives
Expected final shape: `torch.Size([8, 4, 256])`

In [ ]:
vocab_size, output_dim = 50257, 256
torch.manual_seed(123)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer   = torch.nn.Embedding(4, output_dim)   # context_length = 4

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
token_embeddings = token_embedding_layer(inputs)             # (8, 4, 256)
pos_embeddings   = pos_embedding_layer(torch.arange(4))      # (4, 256)
input_embeddings = token_embeddings + pos_embeddings         # broadcast add
print("input_embeddings.shape:", input_embeddings.shape)

That `(8, 4, 256)` tensor is the entire point of this unit — [Unit 3](https://aditya-402.github.io/learn-with-adi/series/llm-from-scratch/unit3.html) feeds it into attention.